In [ ]:
# GPU HARD GATE: stop immediately if the Kaggle session is not using GPU.
import sys
import torch

print('python:', sys.version)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA GPU is not available. Enable GPU accelerator before running.'
print('cuda device count:', torch.cuda.device_count())
print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
# USER PARAMETERS: edit this cell only for smoke, sanity, build-only, fallback, or full training.

RUN_SMOKE_AUDIT = True
FULL_TRAIN = False
BUILD_ONLY = False

# Sanity defaults: 5 epochs x 100 train steps = 500 optimizer steps.
# If FULL_TRAIN=True, train.py ignores validation subset limits and requires full 72-patient validation.
EPOCHS = 5
BATCH_SIZE = 4
MAX_TRAIN_STEPS_PER_EPOCH = 100
MAX_VAL_STEPS_PER_EPOCH = 25
MAX_TRAIN_PATIENTS = None
MAX_VAL_PATIENTS = 8
MAX_SLICES_PER_VAL_PATIENT = 16
NUM_WORKERS = 2
SEED = 731

# Stage 1 default: stem + pos_embed + ResNet stage2 + ViT last4 + decoder.
TRAINABLE_VIT_BLOCKS = 4
TRAINABLE_RESNET_STAGES = '2'
TRAIN_STEM = True
TRAIN_POS_EMBED = True

# Conservative fallback example:
# TRAINABLE_VIT_BLOCKS = 2
# TRAINABLE_RESNET_STAGES = ''

LR_NEW = 3e-4
LR_PRETRAINED = 3e-5
LR_POS_EMBED = 3e-5
WEIGHT_DECAY = 1e-4
BALANCED_TRAIN_SAMPLER = False
TRAIN_SAMPLING_MODE = 'patient_local'  # 'patient_local' or 'random'

# Runtime controls. Use shorter log intervals for benchmark sanity runs.
USE_AMP = True
TRAIN_LOG_EVERY = 500
VAL_LOG_EVERY = 200

# Multi-GPU controls. Start with batch 8 for DataParallel benchmark.
USE_DATA_PARALLEL = False
GPU_IDS = '0,1'

CONFIG = 'configs/pretrained_route_b.yaml'
OUTPUT_DIR = 'run_outputs/route_b_sanity_gpu'
AUDIT_OUTPUT = 'run_outputs/route_b_smoke_audit.json'

# Export controls. Metrics archive is light; checkpoint archive can be large.
EXPORT_RESULTS = True
EXPORT_CHECKPOINT = False
EXPORT_ROOT = '/kaggle/working/exports'

# Kaggle input dataset locations.
CODE_SRC = '/kaggle/input/datasets/mathewzhou/pbl-brain-tumour-code'
DATA_SRC = '/kaggle/input/datasets/mathewzhou/pbl-brain-tumour-training'
WEIGHTS_SRC = '/kaggle/input/datasets/mathewzhou/r50-vit-b16-in21k-pretrained/model.safetensors'

In [ ]:
# Setup a clean Kaggle working copy of the code dataset.
import os
import shutil
import tarfile
from pathlib import Path

CODE_SRC = Path(CODE_SRC)
PROJECT = Path('/kaggle/working/PBL_BrainTumour')

if PROJECT.exists():
    shutil.rmtree(PROJECT)
PROJECT.mkdir(parents=True, exist_ok=True)

def safe_extract_tar(tar_path, destination):
    destination = Path(destination).resolve()
    with tarfile.open(tar_path, 'r:gz') as tar:
        for member in tar.getmembers():
            target = (destination / member.name).resolve()
            try:
                target.relative_to(destination)
            except ValueError as exc:
                raise RuntimeError(f'Unsafe tar member: {member.name}') from exc
        tar.extractall(destination)

if (CODE_SRC / 'src').exists() and (CODE_SRC / 'scripts').exists():
    for item in CODE_SRC.iterdir():
        if item.name.startswith('._') or item.name == '__MACOSX':
            continue
        dst = PROJECT / item.name
        if item.is_dir():
            shutil.copytree(item, dst, ignore=shutil.ignore_patterns('._*', '__MACOSX'))
        else:
            shutil.copy2(item, dst)
elif (CODE_SRC / 'kaggle_code.tar.gz').exists():
    safe_extract_tar(CODE_SRC / 'kaggle_code.tar.gz', PROJECT)
else:
    raise FileNotFoundError(f'Could not find expanded code or kaggle_code.tar.gz in {CODE_SRC}')

os.chdir(PROJECT)
print('cwd:', os.getcwd())
print('project files:', sorted(p.name for p in PROJECT.iterdir()))

In [ ]:
# Create data and pretrained-weight symlinks expected by configs/pretrained_route_b.yaml.
DATA_SRC = Path(DATA_SRC)
WEIGHTS_SRC = Path(WEIGHTS_SRC)

DATA_LINK = PROJECT / 'data/processed/Task01_BrainTumour'
WEIGHTS_LINK = PROJECT / 'models/pretrained/timm/vit_base_r50_s16_224.orig_in21k/model.safetensors'

DATA_LINK.parent.mkdir(parents=True, exist_ok=True)
WEIGHTS_LINK.parent.mkdir(parents=True, exist_ok=True)

if DATA_LINK.exists() or DATA_LINK.is_symlink():
    if DATA_LINK.is_symlink() or DATA_LINK.is_file():
        DATA_LINK.unlink()
    else:
        shutil.rmtree(DATA_LINK)
if WEIGHTS_LINK.exists() or WEIGHTS_LINK.is_symlink():
    WEIGHTS_LINK.unlink()

DATA_LINK.symlink_to(DATA_SRC, target_is_directory=True)
WEIGHTS_LINK.symlink_to(WEIGHTS_SRC)

print('data link:', DATA_LINK, '->', DATA_LINK.resolve())
print('weights link:', WEIGHTS_LINK, '->', WEIGHTS_LINK.resolve())

In [ ]:
# Dependency, GPU, and required-path gate.
import importlib.util
import subprocess

missing = [pkg for pkg in ['timm', 'safetensors', 'nibabel', 'yaml'] if importlib.util.find_spec(pkg) is None]
print('missing packages:', missing)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'safetensors', 'nibabel', 'pyyaml'], check=True)

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA GPU not available'
print('gpu:', torch.cuda.get_device_name(0))

required = [
    PROJECT / 'data/processed/Task01_BrainTumour/imagesTr',
    PROJECT / 'data/processed/Task01_BrainTumour/labelsTr',
    PROJECT / 'src/pretrained_hybrid_vit.py',
    PROJECT / 'scripts/train.py',
    PROJECT / 'scripts/03_route_b_smoke_audit.py',
    PROJECT / 'scripts/04_train_route_b.py',
    PROJECT / CONFIG,
    PROJECT / 'outputs/splits/frozen_patient_split_seed42.csv',
    WEIGHTS_LINK,
]
missing_paths = [str(path) for path in required if not path.exists()]
assert not missing_paths, 'Missing required paths:\n' + '\n'.join(missing_paths)
print('Environment gate passed.')

In [ ]:
# Optional Route B model/interface smoke audit.
def run_command(cmd):
    print('RUN:', ' '.join(str(x) for x in cmd))
    result = subprocess.run(cmd, check=False)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}')

if RUN_SMOKE_AUDIT:
    run_command([sys.executable, 'scripts/03_route_b_smoke_audit.py', '--output', AUDIT_OUTPUT])
else:
    print('Smoke audit skipped.')

In [ ]:
# Build-only, sanity training, fallback training, or full training launcher.
if FULL_TRAIN and not BUILD_ONLY:
    assert MAX_TRAIN_PATIENTS is None, 'Full training must use all training patients.'

cmd = [
    sys.executable,
    'scripts/train.py',
    '--config', CONFIG,
    '--output-dir', OUTPUT_DIR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--seed', str(SEED),
    '--trainable-vit-blocks', str(TRAINABLE_VIT_BLOCKS),
    '--trainable-resnet-stages', str(TRAINABLE_RESNET_STAGES),
    '--lr-new', str(LR_NEW),
    '--lr-pretrained', str(LR_PRETRAINED),
    '--lr-pos-embed', str(LR_POS_EMBED),
    '--weight-decay', str(WEIGHT_DECAY),
    '--gpu-ids', str(GPU_IDS),
    '--train-sampling-mode', str(TRAIN_SAMPLING_MODE),
    '--train-log-every', str(TRAIN_LOG_EVERY),
    '--val-log-every', str(VAL_LOG_EVERY),
]

if FULL_TRAIN:
    cmd.append('--full-train')
else:
    cmd += [
        '--max-train-steps-per-epoch', str(MAX_TRAIN_STEPS_PER_EPOCH),
        '--max-val-steps-per-epoch', str(MAX_VAL_STEPS_PER_EPOCH),
    ]
    if MAX_VAL_PATIENTS is not None:
        cmd += ['--max-val-patients', str(MAX_VAL_PATIENTS)]
    if MAX_SLICES_PER_VAL_PATIENT is not None:
        cmd += ['--max-slices-per-val-patient', str(MAX_SLICES_PER_VAL_PATIENT)]

if MAX_TRAIN_PATIENTS is not None:
    cmd += ['--max-train-patients', str(MAX_TRAIN_PATIENTS)]
if BUILD_ONLY:
    cmd.append('--build-only')

cmd.append('--train-stem' if TRAIN_STEM else '--no-train-stem')
cmd.append('--train-pos-embed' if TRAIN_POS_EMBED else '--no-train-pos-embed')
cmd.append('--balanced-train-sampler' if BALANCED_TRAIN_SAMPLER else '--no-balanced-train-sampler')
if USE_AMP:
    cmd.append('--amp')
if USE_DATA_PARALLEL:
    cmd.append('--data-parallel')

run_command(cmd)


In [ ]:
# Results display. BUILD_ONLY produces metadata but no loss history.
import pandas as pd

history_path = Path(OUTPUT_DIR) / 'loss_history.csv'
metadata_path = Path(OUTPUT_DIR) / 'training_metadata.json'

if history_path.exists():
    history = pd.read_csv(history_path)
    display(history)
    print('loss curve:', PROJECT / OUTPUT_DIR / 'loss_curves.png')
    print('checkpoint:', PROJECT / OUTPUT_DIR / 'best_checkpoint.pt')
else:
    print('No training history generated.')

if metadata_path.exists():
    print('metadata:', PROJECT / metadata_path)
if RUN_SMOKE_AUDIT:
    print('audit:', PROJECT / AUDIT_OUTPUT)

In [ ]:
# Export this run's results for download. Metrics archive is always light; full archive is optional.
import os
import tarfile

if EXPORT_RESULTS:
    RUN_DIR = PROJECT / OUTPUT_DIR
    assert RUN_DIR.exists(), f'Run directory not found: {RUN_DIR}'
    EXPORT_ROOT = Path(EXPORT_ROOT)
    EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

    light_files = [
        'loss_history.csv',
        'training_metadata.json',
        'loss_curves.png',
        'validation_metrics.csv',
    ]
    metrics_archive = EXPORT_ROOT / f'{Path(OUTPUT_DIR).name}_metrics.tar.gz'
    with tarfile.open(metrics_archive, 'w:gz') as tar:
        added = 0
        for name in light_files:
            path = RUN_DIR / name
            if path.exists():
                tar.add(path, arcname=f'{RUN_DIR.name}/{name}')
                added += 1
    print('Metrics export created:')
    print(metrics_archive)
    print(f'Size: {os.path.getsize(metrics_archive) / 1024**2:.2f} MB')
    print(f'Files added: {added}')

    if EXPORT_CHECKPOINT:
        full_archive = shutil.make_archive(
            str(EXPORT_ROOT / RUN_DIR.name),
            'gztar',
            root_dir=RUN_DIR.parent,
            base_dir=RUN_DIR.name,
        )
        print('Full run export created:')
        print(full_archive)
        print(f'Size: {os.path.getsize(full_archive) / 1024**2:.2f} MB')
    else:
        print('Full checkpoint export skipped. Set EXPORT_CHECKPOINT = True to include .pt files.')
else:
    print('Export skipped.')
